**Model Evaluation**

15) Evaluate ALL models with 5 metrics

In [ ]:
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.sql import functions as F
import pandas as pd

eval_metrics = ["accuracy", "weightedPrecision", "weightedRecall", "f1"]

def macro_f1_from_preds(pred_df, label_col="label", pred_col="prediction"):
    # Get label values that actually appear in this evaluation set
    label_list = [float(r[0]) for r in pred_df.select(label_col).distinct().collect()]

    rdd = pred_df.select(pred_col, label_col).rdd.map(lambda r: (float(r[0]), float(r[1])))
    metrics = MulticlassMetrics(rdd)

    f1s = [metrics.fMeasure(l) for l in label_list]
    return float(sum(f1s) / len(f1s)) if f1s else 0.0

comparison_rows = []

# Add tuned model as well
all_models = dict(trained_pipes)
all_models["LR_TUNED"] = best_lr_model.bestModel

for model_name, model_obj in all_models.items():
    preds = model_obj.transform(test_df_ckpt)

    row = {"Model": model_name}
    for m in eval_metrics:
        row[m] = float(evaluator.setMetricName(m).evaluate(preds))

    # 5th metric
    row["macroF1"] = macro_f1_from_preds(preds)

    comparison_rows.append(row)

comparison_pdf = pd.DataFrame(comparison_rows).sort_values("f1", ascending=False)
comparison_pdf.to_csv("all_models_metrics_5.csv", index=False)

print("Saved: all_models_metrics_5.csv")
comparison_pdf

16) NOVA-4 recall per model

In [ ]:
# Extract the trained StringIndexer from the best tuned LR pipeline
# This shows how original nova_group values were mapped to numeric labels
indexer_model = best_lr_model.bestModel.stages[0]
labels_order = indexer_model.labels  # original nova_group values as strings
print("StringIndexer label order:", labels_order)

# Identify which numeric label corresponds to NOVA group 4
# This is needed because Spark internally converts class labels to indices
nova4_label_value = None
if "4" in labels_order:
    nova4_label_value = float(labels_order.index("4"))
print("Label index for NOVA=4:", nova4_label_value)


# Function to calculate recall for a specific class
# Uses Spark's MulticlassMetrics on prediction output
def per_class_recall(pred_df, target_label, label_col="label", pred_col="prediction"):
    rdd = pred_df.select(pred_col, label_col).rdd.map(lambda r: (float(r[0]), float(r[1])))
    metrics = MulticlassMetrics(rdd)
    return float(metrics.recall(float(target_label)))


# Compute recall for NOVA group 4 across all trained models
# This helps evaluate how well each model detects ultra-processed foods
rows = []
if nova4_label_value is not None:
    for model_name, model_obj in all_models.items():
        preds = model_obj.transform(test_df_ckpt)
        rows.append((model_name, per_class_recall(preds, nova4_label_value)))

# Save results for reporting and Tableau dashboard
nova4_df = pd.DataFrame(rows, columns=["Model", "Recall_NOVA4"])
nova4_df.to_csv("nova4_recall_by_model.csv", index=False)
print("Saved: nova4_recall_by_model.csv")

nova4_df

17) Bootstrap Confidence Interval for best model (percentile CI)

In [ ]:
# Identify the best performing model based on the comparison table
best_model_name = comparison_pdf.iloc[0]["Model"]
best_model = all_models[best_model_name]

# Generate predictions from the best model on the test dataset
# Only keep label and prediction columns for evaluation
best_preds = best_model.transform(test_df_ckpt).select("label", "prediction").cache()

# Bootstrap settings
# B defines the number of resampling iterations
B = 50
scores = []

# Perform bootstrap resampling on prediction results
# Each iteration samples with replacement and calculates F1 score
for i in range(B):
    sample = best_preds.sample(withReplacement=True, fraction=1.0, seed=100+i)
    scores.append(float(evaluator.setMetricName("f1").evaluate(sample)))

# Convert results to numpy array for statistical calculations
scores = np.array(scores)

# Compute 95% confidence interval using percentiles
ci_low = float(np.percentile(scores, 2.5))
ci_high = float(np.percentile(scores, 97.5))

# Store bootstrap summary statistics
ci_df = pd.DataFrame([{
    "Model": best_model_name,
    "Metric": "f1",
    "Mean": float(scores.mean()),
    "Std": float(scores.std()),
    "CI_2.5": ci_low,
    "CI_97.5": ci_high,
    "B": B
}])

# Save results for reporting
ci_df.to_csv("bootstrap_ci_bestmodel.csv", index=False)
print("Saved: bootstrap_ci_bestmodel.csv")

ci_df